# 🚀 SmartCity Data Simulator

This notebook generates synthetic Smart City operational datasets.

Datasets Generated:

- Bus GPS
- Emergency Incidents

The generated datasets intentionally contain configurable anomalies to test the AI Self-Healing ETL Pipeline.

## Import Libraries

In [0]:
import json
import random

from datetime import datetime, timedelta

import pandas as pd

## Load Configuration

In [0]:
display(dbutils.fs.ls("dbfs:/"))

In [0]:
# Import configuration

from utils.config import *

print("Configuration Loaded Successfully ✅")

print(f"Bus Records       : {BUS_RECORDS}")
print(f"Emergency Records : {EMERGENCY_RECORDS}")
print(f"Random Seed       : {RANDOM_SEED}")
print(f"Zones             : {ZONES}")

## Generate One Bus Record

In [0]:
def generate_bus_record():

    record = {
        "bus_id": f"BUS{random.randint(100,999)}",
        "route_id": random.choice(ROUTES),
        "zone": random.choice(ZONES),
        "latitude": round(random.uniform(17.20,17.60),6),
        "longitude": round(random.uniform(78.20,78.70),6),
        "speed_kmh": random.randint(0,80),
        "delay_minutes": random.randint(0,20),
        "occupancy": random.randint(0,60),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return record

In [0]:
bus = generate_bus_record()

print(bus)

## Generate Bus Dataset

In [0]:
bus_records = []

for _ in range(BUS_RECORDS):
    bus_records.append(generate_bus_record())

print(f"Generated {len(bus_records)} bus records.")

In [0]:
bus_df = pd.DataFrame(bus_records)

bus_df.head()

## Generate Emergency Record

In [0]:
def generate_emergency_record():

    record = {
        "incident_id": f"INC{random.randint(10000,99999)}",
        "zone": random.choice(ZONES),
        "incident_type": random.choice(INCIDENT_TYPES),
        "severity": random.randint(1,5),
        "response_time": random.randint(3,25),
        "status": random.choice([
            "Open",
            "Closed",
            "In Progress"
        ]),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return record

In [0]:
emergency = generate_emergency_record()

print(emergency)

In [0]:
emergency_records = []

for _ in range(EMERGENCY_RECORDS):
    emergency_records.append(generate_emergency_record())

print(f"Generated {len(emergency_records)} emergency records.")

In [0]:
emergency_df = pd.DataFrame(emergency_records)

emergency_df.head()

## Save Clean Datasets

In [0]:
from datetime import datetime

BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Batch ID:", BATCH_ID)

In [0]:
import os

PROJECT_ROOT = os.path.abspath("../../")

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "Data",
    "raw",
    BATCH_ID
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Output Folder:", OUTPUT_DIR)

## Data Quality Test Scenario Generation

In [0]:
TEST_SCENARIOS = [
    {
        "name": "missing_zone",
        "description": "Missing zone information"
    },
    {
        "name": "invalid_gps",
        "description": "Latitude or longitude outside valid range"
    },
    {
        "name": "negative_delay",
        "description": "Negative bus delay value"
    },
    {
        "name": "future_timestamp",
        "description": "Timestamp later than current time"
    },
    {
        "name": "duplicate_record",
        "description": "Duplicate vehicle identifier"
    }
]

## Test Scenario Engine

In [0]:
def inject_test_scenario(df, scenario_name):

    print(f"Applying test scenario: {scenario_name}")

    if scenario_name == "missing_zone":
        return inject_missing_zone(df)

    elif scenario_name == "invalid_gps":
        return inject_invalid_gps(df)

    elif scenario_name == "negative_delay":
        return inject_negative_delay(df)

    elif scenario_name == "future_timestamp":
        return inject_future_timestamp(df)
    
    elif scenario_name == "duplicate_record":
        return inject_duplicate_record(df)

    return df

## Test Scenario Helper Functions

In [0]:
def get_random_sample(df):
    df = df.copy()
    sample_size = int(len(df) * (ANOMALY_PERCENTAGE / 100))
    random_index = random.sample(
        list(df.index),
        sample_size
    )
    return df, random_index, sample_size

In [0]:
def inject_missing_zone(df):

    df, random_index, sample_size = get_random_sample(df)

    df.loc[random_index, "zone"] = None

    print(f"✓ Missing Zone injected into {sample_size} records.")

    return df

In [0]:
def inject_invalid_gps(df):
    df, random_index, sample_size = get_random_sample(df)
    gps_variants = [
        "invalid_latitude",
        "invalid_longitude",
        "invalid_both"
    ]
    for idx in random_index:
        variant = random.choice(gps_variants)
        if variant == "invalid_latitude":
            df.loc[idx, "latitude"] = random.uniform(100, 200)
        elif variant == "invalid_longitude":
            df.loc[idx, "longitude"] = random.uniform(200, 300)
        else:
            df.loc[idx, "latitude"] = random.uniform(100, 200)
            df.loc[idx, "longitude"] = random.uniform(200, 300)
    print(f"✓ Invalid GPS injected into {sample_size} records.")
    return df

In [0]:
def inject_negative_delay(df):
    df, random_index, sample_size = get_random_sample(df)
    for idx in random_index:
        df.loc[idx, "delay_minutes"] = -random.randint(1, 30)
    print(f"✓ Negative Delay injected into {sample_size} records.")
    return df

In [0]:
def inject_future_timestamp(df):
    df, random_index, sample_size = get_random_sample(df)
    for idx in random_index:
        future_time = datetime.now() + timedelta(days=random.randint(1,30))
        df.loc[idx, "timestamp"] = future_time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"✓ Future Timestamp injected into {sample_size} records.")
    return df

In [0]:
def inject_duplicate_record(df):
    df, random_index, sample_size = get_random_sample(df)
    duplicate_rows = df.loc[random_index].copy()
    df = pd.concat(
        [df, duplicate_rows],
        ignore_index=True
    )
    print(f"✔ Duplicate Records injected: {sample_size}")
    return df

In [0]:
print("=" * 60)
print("Injecting all anomalies into Bus GPS dataset...")
print("=" * 60)

bus_df = inject_test_scenario(bus_df, "missing_zone")
bus_df = inject_test_scenario(bus_df, "invalid_gps")
bus_df = inject_test_scenario(bus_df, "negative_delay")
bus_df = inject_test_scenario(bus_df, "future_timestamp")
bus_df = inject_test_scenario(bus_df, "duplicate_record")

print("=" * 60)
print("✅ All anomalies injected successfully!")
print("Total Records:", len(bus_df))
print("=" * 60)

In [0]:
bus_df.to_json(
    os.path.join(
        OUTPUT_DIR,
        "bus_gps_clean.json"
    ),
    orient="records",
    indent=4
)

print("✅ Bus GPS dataset saved with anomalies.")

# Emergency Incident Data Quality Test Generation

## Emergency Test Scenario Helper Functions

In [0]:
def get_random_sample(df):
    df = df.copy()
    sample_size = int(len(df) * (ANOMALY_PERCENTAGE / 100))
    random_index = random.sample(
        list(df.index),
        sample_size
    )
    return df, random_index, sample_size

In [0]:
def inject_missing_zone_emergency(df):
    df, random_index, sample_size = get_random_sample(df)
    df.loc[random_index, "zone"] = None
    print(f"✔ Missing Zone injected into {sample_size} records.")
    return df

In [0]:
def inject_invalid_severity(df):
    df, random_index, sample_size = get_random_sample(df)
    for idx in random_index:
        df.loc[idx, "severity"] = random.choice([-1, 0, 99])
    print(f"✔ Invalid Severity injected into {sample_size} records.")
    return df

In [0]:
def inject_negative_response_time(df):
    df, random_index, sample_size = get_random_sample(df)
    for idx in random_index:
        df.loc[idx, "response_time"] = -random.randint(1,30)
    print(f"✔ Negative Response Time injected into {sample_size} records.")
    return df

In [0]:
from datetime import timedelta
def inject_future_timestamp_emergency(df):
    df, random_index, sample_size = get_random_sample(df)
    for idx in random_index:
        future_time = datetime.now() + timedelta(days=random.randint(1,30))
        df.loc[idx, "timestamp"] = future_time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"✔ Future Timestamp injected into {sample_size} records.")
    return df

In [0]:
def inject_duplicate_record_emergency(df):
    df, random_index, sample_size = get_random_sample(df)
    duplicate_rows = df.loc[random_index].copy()
    df = pd.concat(
        [df, duplicate_rows],
        ignore_index=True
    )
    print(f"✔ Duplicate Records injected: {sample_size}")
    return df

## Emergency Test Scenario Engine

In [0]:
print("=" * 60)
print("Injecting all anomalies into Emergency dataset...")
print("=" * 60)
emergency_df = inject_missing_zone_emergency(emergency_df)
emergency_df = inject_invalid_severity(emergency_df)
emergency_df = inject_negative_response_time(emergency_df)
emergency_df = inject_future_timestamp_emergency(emergency_df)
emergency_df = inject_duplicate_record_emergency(emergency_df)
print("=" * 60)
print("✅ All Emergency anomalies injected successfully!")
print("Total Records:", len(emergency_df))
print("=" * 60)

## Save Emergency Dataset

In [0]:
emergency_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "emergency_clean.csv"
    ),
    index=False
)
print("✅ Emergency dataset saved with anomalies.")

## Verify Emergency Dataset

In [0]:
check_emergency_df = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "emergency_clean.csv"
    )
)
print("Rows :", len(check_emergency_df))
print(
    "Missing Zone :",
    check_emergency_df["zone"].isna().sum()
)
print(
    "Invalid Severity :",
    (~check_emergency_df["severity"].between(1,5)).sum()
)
print(
    "Negative Response Time :",
    (check_emergency_df["response_time"] < 0).sum()
)
print(
    "Future Timestamp :",
    (
        pd.to_datetime(check_emergency_df["timestamp"])
        > pd.Timestamp.now()
    ).sum()
)
print(
    "Duplicate Records :",
    check_emergency_df.duplicated().sum()
)

## verify 

In [0]:
check_bus_df = pd.read_json(
    os.path.join(
        OUTPUT_DIR,
        "bus_gps_clean.json"
    )
)
print("Rows :", len(check_bus_df))
print("Missing Zone      :", check_bus_df["zone"].isna().sum())
print(
    "Invalid GPS       :",
    (
        (check_bus_df["latitude"] > 90)
        | (check_bus_df["latitude"] < -90)
        | (check_bus_df["longitude"] > 180)
        | (check_bus_df["longitude"] < -180)
    ).sum()
)
print(
    "Negative Delay    :",
    (check_bus_df["delay_minutes"] < 0).sum()
)
print(
    "Future Timestamp  :",
    (
        pd.to_datetime(check_bus_df["timestamp"])
        > pd.Timestamp.now()
    ).sum()
)
print(
    "Duplicate Records :",
    check_bus_df.duplicated().sum()
)